# Queen Editor — Colab Host

**Input:** Private repo `AltanBaysal/Internal-tools` (dal: `prod`) | **Output:** cloudflared tüneliyle açılan Queen Editor arayüzü

Bütün iş repo'daki `queen-editor/colab/host.py`'de: build + serve + cloudflared tünel. Bu notebook sadece repoyu çeker ve onu çağırır.

**Tek hazırlık:** Sol menü 🔑 **Secrets** → `GITHUB_TOKEN` ekle (GitHub fine-grained token, sadece `Internal-tools`, **Contents: Read**), "Notebook access" açık.

Sonra **Runtime → Run all**. Çıkan `🌐 https://...trycloudflare.com` linkini aç.

In [ ]:
# === Bootstrap: token (Secrets) -> clone/pull prod ===
import os, subprocess
from google.colab import userdata

DEST = "/content/Internal-tools"
TOKEN = userdata.get("GITHUB_TOKEN")
assert TOKEN, "❌ GITHUB_TOKEN Secrets'ta yok — 🔑 Secrets'tan ekle (Contents: Read)"

if not os.path.exists(DEST):
    url = f"https://{TOKEN}@github.com/AltanBaysal/Internal-tools.git"
    subprocess.run(["git", "clone", "--depth", "1", "-b", "prod", "--single-branch", url, DEST], check=True)
else:
    subprocess.run(["git", "-C", DEST, "pull"], check=True)
print("✅ Repo hazır:", DEST)

In [ ]:
# === Kurulum: cloudflared + frontend build (loglar bu hücrede) ===
import sys
sys.path.insert(0, f"{DEST}/queen-editor/colab")
import host

host.ensure_cloudflared()
host.build()

In [ ]:
# === Çalıştır: tünel + serve (canlı — cloudflared logları + URL burada akar) ===
# Bu hücre açık kaldıkça tünel yaşar. Durdurmak için cell'i durdur.
host.serve_with_tunnel()